In [1]:

# Homework Day 1 - Cell 1: Install Dependencies

%pip install -q transformers sentencepiece



In [7]:

# Homework Day 1 - Cell 2: Load the Real Model

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "google/flan-t5-small"
MODEL_REVISION = "0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab"

torch.set_num_threads(2)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
    trust_remote_code=False
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
    trust_remote_code=False,
    use_safetensors=True
).to("cpu").eval()

print("Model loaded successfully!")
print("Model name:", MODEL_NAME)


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model loaded successfully!
Model name: google/flan-t5-small


In [8]:

# Homework Day 1 - Cell 3: Generate an Answer

import torch
import time

context = "You can check your application status using its reference number."
question = "How can I check my application status?"

prompt = (
    f"Answer the question using the context.\n"
    f"Context: {context}\n"
    f"Question: {question}\n"
    f"Answer:"
)

encoded = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=256
)

start = time.perf_counter()

with torch.inference_mode():
    generated = model.generate(
        **encoded,
        max_new_tokens=48,
        do_sample=False,
        num_beams=1
    )

inference_seconds = round(time.perf_counter() - start, 3)

answer = tokenizer.decode(
    generated[0],
    skip_special_tokens=True
).strip()

print("Question:", question)
print("Generated Answer:", answer)
print("Inference Time:", inference_seconds, "seconds")
print("Input Tokens:", encoded["input_ids"].shape[-1])
print("Output Tokens:", generated.shape[-1])


Question: How can I check my application status?
Generated Answer: Use its reference number
Inference Time: 2.61 seconds
Input Tokens: 34
Output Tokens: 6


In [9]:

# Homework Day 1 - Cell 4: Save and Verify Report

import json
from pathlib import Path

MODEL_REVISION = "0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab"

OUTPUT_DIR = Path("murshid_live_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

report = {
    "mode": "real_local_model",
    "model": MODEL_NAME,
    "model_revision": MODEL_REVISION,
    "device": "cpu",
    "context": context,
    "question": question,
    "answer": answer,
    "inference_seconds": inference_seconds,
    "input_tokens": int(encoded["input_ids"].shape[-1]),
    "output_tokens_including_special": int(generated.shape[-1]),
    "external_inference_api_calls": 0,
    "review_required": (
        "Check whether the answer tells the user to use the reference number. "
        "Reject unsupported fees, dates or status claims."
    )
}

output_path = OUTPUT_DIR / "live_demo_result.json"

output_path.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

assert report["answer"] == "Use its reference number"
assert report["input_tokens"] == 34
assert report["output_tokens_including_special"] == 6

print(json.dumps(report, indent=2))
print("\nComparison with expected: PASSED")
print("Saved to:", output_path)


{
  "mode": "real_local_model",
  "model": "google/flan-t5-small",
  "model_revision": "0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab",
  "device": "cpu",
  "context": "You can check your application status using its reference number.",
  "question": "How can I check my application status?",
  "answer": "Use its reference number",
  "inference_seconds": 2.61,
  "input_tokens": 34,
  "output_tokens_including_special": 6,
  "external_inference_api_calls": 0,
  "review_required": "Check whether the answer tells the user to use the reference number. Reject unsupported fees, dates or status claims."
}

Comparison with expected: PASSED
Saved to: murshid_live_outputs/live_demo_result.json


### Explanation

1. The input context and question are converted into tokens by the tokenizer and passed to the FLAN-T5 model to generate an answer.
2. Unlike FakeClient, which returns predefined responses, the real model generates output tokens that are decoded into the final answer.
